# Notebook 2: Video-Level Min/Max Feature Extraction

The purpose of this notebook is to prepare a video-level dataset by extracting minimum and maximum values from selected sensor and expression columns.

The extraction is performed using whole-video windowing, meaning that all available rows for one participant and one video are grouped together. For each selected feature column, two values are calculated: the minimum value and the maximum value during the whole video.

The final dataset will later be merged with the corresponding valence and arousal labels and saved as:

`video_level_valence_arousal_minmax.csv`


## 1. Import Required Libraries

In this section, the required Python libraries are imported.


In [56]:
import pandas as pd
import numpy as np
from pathlib import Path #za polesno da pristapam do folderite so podatoci
import re 

## 2. Data Loading and Initial Folder Inspection

The dataset used in this notebook is stored locally on an external disk. Before starting the feature extraction process, the main data directory is defined and checked to confirm that the files are accessible from Python.

In this section, only a brief inspection of the folder structure is performed. The goal is to verify that the dataset path is correct, list the available participant folders, and collect the CSV files that will be used in the next steps.

A more detailed analysis of the dataset structure, participant information, available columns, missing values, and labels is performed in `01_data_understanding.ipynb`. Therefore, this notebook includes only the necessary loading and inspection steps required to continue with video-level feature extraction.


In [57]:
data_dir = Path(r"E:\SSAI_structured_data\data3")

print(data_dir.exists())
print(data_dir)

True
E:\SSAI_structured_data\data3


In [58]:
for f in data_dir.iterdir():
    print(f.name)

final_data_details.csv
final_data_questionnaires.csv
participant1
participant10
participant100
participant101
participant102
participant103
participant104
participant105
participant11
participant12-1
participant12-2
participant13
participant14
participant15
participant16
participant17
participant18
participant19
participant2
participant20
participant21
participant22
participant23
participant24
participant25
participant26
participant27
participant28
participant29
participant3
participant30
participant31
participant32
participant33
participant34
participant35
participant36
participant37
participant38
participant39
participant4
participant40
participant41
participant42
participant43
participant44
participant45
participant46
participant47
participant48
participant49
participant5
participant50
participant51
participant52
participant53
participant54
participant55
participant56
participant57
participant58
participant59
participant6
participant60
participant61
participant62
participant63
parti

In [59]:
participant_folders = [f for f in data_dir.iterdir() if f.is_dir()]

print("Number of participants:", len(participant_folders))

for folder in participant_folders[:10]:  
    print(folder.name)

Number of participants: 106
participant1
participant10
participant100
participant101
participant102
participant103
participant104
participant105
participant11
participant12-1


In [60]:
csv_files = []

for folder in participant_folders:
    csv_files.extend(folder.rglob("*.csv"))

print("Total number of CSV files (from all participants):", len(csv_files))


Total number of CSV files (from all participants): 1007


## 3. Participant and File Identification

This section extracts participant identifiers from folder names and creates an overview of all CSV files associated with each participant.

In [61]:
# vaka ke gi stavi 12-1 i 12-2 u ist folder
def extract_participant_id_(folder_name):
    match = re.search(r"participant(\d+)", folder_name)

    if match:
        return int(match.group(1)) # 1, zarai participant 12 sto ima 1 i 2
    else:
        return None

In [62]:
#vaka gi stava 12-1 i 12-2 vo razlicen folder, tak sto id 12 ke e za 12-1, a id na so posleden br ke e 12-2
#so cel da nema problem so podatocite ako se vo isf folder, kako duplikati se
#ova mora da vidime kako ke go rabotime!!

def extract_participant_id(folder_name):
    match = re.search(r"participant(\d+)(?:-(\d+))?", folder_name)

    if match:
        participant_id = int(match.group(1))
        participant_part = match.group(2)

        if participant_id == 12 and participant_part == "2":
            return 106

        return participant_id

    return None


In [63]:
#just for testing the function

test_folders = [
    "participant1",
    "participant12-1",
    "participant12-2",
    "participant105"
]

for folder_name in test_folders:
    print(folder_name, "->", extract_participant_id(folder_name))

participant1 -> 1
participant12-1 -> 12
participant12-2 -> 106
participant105 -> 105


In [64]:
#sortiranje na papkite

participant_folders = sorted(
    participant_folders,
    key = lambda folder: (
        extract_participant_id(folder.name),
        folder.name
    )
)

for folder in participant_folders[:15]:
    print(folder.name, "-> ID:", extract_participant_id(folder.name))

participant1 -> ID: 1
participant2 -> ID: 2
participant3 -> ID: 3
participant4 -> ID: 4
participant5 -> ID: 5
participant6 -> ID: 6
participant7 -> ID: 7
participant8 -> ID: 8
participant9 -> ID: 9
participant10 -> ID: 10
participant11 -> ID: 11
participant12-1 -> ID: 12
participant13 -> ID: 13
participant14 -> ID: 14
participant15 -> ID: 15


In [65]:
#table with csv files

file_records = []

for folder in participant_folders:
    participant_id = extract_participant_id(folder.name)

    for file_path in folder.rglob("*.csv"):
        file_records.append({
            "participant_id": participant_id,
            "participant_folder": folder.name,
            "file_name": file_path.name,
            "file_path": str(file_path)
        })

files_df = pd.DataFrame(file_records)

print("Shape:", files_df.shape)
files_df.head(20)

Shape: (1007, 4)


,participant_id,participant_folder,file_name,file_path
0,1,participant1,3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\3NV...
1,1,participant1,arousal_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\aro...
2,1,participant1,breathingRate_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\bre...
3,1,participant1,emgActivation_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\emg...
4,1,participant1,expression_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\exp...
5,1,participant1,facialActivation_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\fac...
6,1,participant1,facialValence_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\fac...
7,1,participant1,headMotion_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\hea...
8,1,participant1,hrv_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\hrv...
9,1,participant1,valence_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\val...


In [66]:
files_df["file_name"].value_counts()

files_df.groupby("participant_folder").size().value_counts().sort_index()

6      1
8     22
9      5
10    78
Name: count, dtype: int64

In [67]:
files_per_folder_summary = (
    files_df
    .groupby("participant_folder")
    .size()
    .value_counts()
    .sort_index()
    .reset_index()
)

files_per_folder_summary.columns = [
    "number_of_csv_files_in_folder",
    "number_of_participant_folders"
]

files_per_folder_summary

,number_of_csv_files_in_folder,number_of_participant_folders
0,6,1
1,8,22
2,9,5
3,10,78


The results show that not all participant folders contain the same number of CSV files. Most participant folders contain 10 CSV files, but some folders contain fewer files.

Because of this, the next step is to identify which participant folders have missing files and which specific file types are missing. This is important before feature extraction, because the final dataset should be created only from the available and correctly identified CSV files.

In [68]:
files_per_folder = (
    files_df
    .groupby("participant_folder")
    .size()
    .reset_index(name="number_of_files")
)

files_per_folder.sort_values("number_of_files").head(30)

,participant_folder,number_of_files
75,participant71,6
10,participant12-2,8
26,participant27,8
31,participant31,8
58,participant56,8
59,participant57,8
57,participant55,8
50,participant49,8
48,participant47,8
49,participant48,8


In [69]:
folders_with_less_than_10_files = (
    files_per_folder[files_per_folder["number_of_files"] < 10]
    .sort_values("number_of_files")
    .reset_index(drop=True)
)

print("Number of participant folders with fewer than 10 CSV files:", len(folders_with_less_than_10_files))

folders_with_less_than_10_files

Number of participant folders with fewer than 10 CSV files: 28


,participant_folder,number_of_files
0,participant71,6
1,participant12-2,8
2,participant31,8
3,participant27,8
4,participant34,8
5,participant33,8
6,participant39,8
7,participant32,8
8,participant50,8
9,participant44,8


Some participant folders contain fewer CSV files than others. Since the detailed completeness analysis belongs to the data understanding notebook, this notebook only keeps this as a practical check before feature extraction.

The next step is to identify the CSV file type for each file, so that only the relevant sensor and expression files can be selected for video-level min/max feature extraction.

In [73]:
def extract_file_type(file_name):
    known_file_types = [
        "arousal",
        "breathingRate",
        "emgActivation",
        "expression",
        "facialActivation",
        "facialValence",
        "headMotion",
        "hrv",
        "valence"

    ]
    for file_type in known_file_types:
        if file_type in file_name:
            return file_type
        
    return "raw_sensor_data"

In [74]:
files_df["file_type"] = files_df["file_name"].apply(extract_file_type)

files_df[["participant_id", "participant_folder", "file_name", "file_type"]].head(20)

,participant_id,participant_folder,file_name,file_type
0,1,participant1,3NVLIWA4.csv,raw_sensor_data
1,1,participant1,arousal_3NVLIWA4.csv,arousal
2,1,participant1,breathingRate_3NVLIWA4.csv,breathingRate
3,1,participant1,emgActivation_3NVLIWA4.csv,emgActivation
4,1,participant1,expression_3NVLIWA4.csv,expression
5,1,participant1,facialActivation_3NVLIWA4.csv,facialActivation
6,1,participant1,facialValence_3NVLIWA4.csv,facialValence
7,1,participant1,headMotion_3NVLIWA4.csv,headMotion
8,1,participant1,hrv_3NVLIWA4.csv,hrv
9,1,participant1,valence_3NVLIWA4.csv,valence


In [75]:
file_type_summary = (
    files_df["file_type"]
    .value_counts()
    .reset_index()
)

file_type_summary.columns = ["file_type", "number_of_files"]

file_type_summary

,file_type,number_of_files
0,emgActivation,106
1,breathingRate,106
2,facialValence,106
3,hrv,106
4,expression,106
5,facialActivation,106
6,valence,105
7,arousal,105
8,raw_sensor_data,84
9,headMotion,77


The file type summary shows that most derived CSV file types are available for almost all participant folders. However, some file types, such as raw sensor data and head motion, are available for fewer participants.

Since this notebook focuses on video-level min/max feature extraction, this information is used only as an availability check. The next step is to inspect the structure and columns of each CSV file type in order to decide which numeric columns can be used for feature extraction.

## 4. CSV Structure Inspection

In this section, one sample CSV file from each file type is opened in order to inspect its structure.

The goal is to identify:
- which columns are available in each file type;
- which columns are numeric;
- which columns can be used for video-level minimum and maximum feature extraction;
- whether each file contains time-related columns needed for whole-video windowing.

Only a small number of rows is loaded during this inspection, because some CSV files are very large.

In [90]:
from IPython.display import display

In [91]:
def read_csv_auto(path, nrows=None):
    path = Path(path)

    skip_rows = 0

    with open(path, "r", encoding="latin1") as file:
        for line in file:
            if line.startswith("#"):
                skip_rows += 1
            else:
                break

    separators = [",", ";", "\t"]
    encodings = ["utf-8-sig", "utf-8", "latin1"]

    best_df = None

    for encoding in encodings:
        for sep in separators:
            try:
                df = pd.read_csv(
                    path,
                    sep=sep,
                    encoding=encoding,
                    skiprows=skip_rows,
                    nrows=nrows,
                    low_memory=False
                )

                if best_df is None or df.shape[1] > best_df.shape[1]:
                    best_df = df

            except Exception:
                pass

    if best_df is not None:
        return best_df

    return pd.read_csv(
        path,
        encoding="latin1",
        skiprows=skip_rows,
        nrows=nrows,
        on_bad_lines="skip",
        low_memory=False
    )

In [92]:
sample_files_by_type = (
    files_df
    .sort_values(["file_type", "participant_id"])
    .groupby("file_type")
    .first()
    .reset_index()
)

sample_files_by_type[[
    "file_type",
    "participant_id",
    "participant_folder",
    "file_name",
    "file_path"
]]

,file_type,participant_id,participant_folder,file_name,file_path
0,arousal,1,participant1,arousal_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\aro...
1,breathingRate,1,participant1,breathingRate_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\bre...
2,emgActivation,1,participant1,emgActivation_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\emg...
3,expression,1,participant1,expression_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\exp...
4,facialActivation,1,participant1,facialActivation_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\fac...
5,facialValence,1,participant1,facialValence_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\fac...
6,headMotion,1,participant1,headMotion_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\hea...
7,hrv,1,participant1,hrv_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\hrv...
8,raw_sensor_data,1,participant1,3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\3NV...
9,valence,1,participant1,valence_3NVLIWA4.csv,E:\SSAI_structured_data\data3\participant1\val...


In [93]:
#Compact inspection table
csv_structure_records = []

for _, row in sample_files_by_type.iterrows():
    file_type = row["file_type"]
    file_path = Path(row["file_path"])
    
    df_sample = read_csv_auto(file_path, nrows=1000)
    
    numeric_columns = df_sample.select_dtypes(include=np.number).columns.tolist()
    
    possible_time_columns = [
        col for col in df_sample.columns
        if any(keyword in str(col).lower() for keyword in ["time", "timestamp", "frame"])
    ]
    
    csv_structure_records.append({
        "file_type": file_type,
        "sample_file": row["file_name"],
        "rows_loaded": df_sample.shape[0],
        "number_of_columns": df_sample.shape[1],
        "number_of_numeric_columns": len(numeric_columns),
        "possible_time_columns": possible_time_columns,
        "first_numeric_columns": numeric_columns[:10]
    })

csv_structure_df = pd.DataFrame(csv_structure_records)

csv_structure_df

,file_type,sample_file,rows_loaded,number_of_columns,number_of_numeric_columns,possible_time_columns,first_numeric_columns
0,arousal,arousal_3NVLIWA4.csv,1000,6,6,"[Frame#, Time]","[Frame#, Time, Arousal/class, Arousal/probabil..."
1,breathingRate,breathingRate_3NVLIWA4.csv,1000,5,5,"[Frame#, Time]","[Frame#, Time, BreathingRate, Ppg/QualityIndex..."
2,emgActivation,emgActivation_3NVLIWA4.csv,1000,6,6,"[Frame#, Time]","[Frame#, Time, Emg/Amplitude/zygo/weighted, Em..."
3,expression,expression_3NVLIWA4.csv,1000,4,3,"[Frame#, Time]","[Frame#, Time, Expression/Intensity]"
4,facialActivation,facialActivation_3NVLIWA4.csv,1000,3,3,"[Frame#, Time]","[Frame#, Time, FacialActivation]"
5,facialValence,facialValence_3NVLIWA4.csv,1000,3,3,"[Frame#, Time]","[Frame#, Time, FacialValence]"
6,headMotion,headMotion_3NVLIWA4.csv,2,1,1,[],[headMotion]
7,hrv,hrv_3NVLIWA4.csv,1000,9,9,"[Frame#, Time]","[Frame#, Time, HRV/mean_hr, HRV/rr, HRV/sdnn, ..."
8,raw_sensor_data,3NVLIWA4.csv,1000,59,59,"[Frame#, Time]","[Frame#, Time, Faceplate/FaceState, Faceplate/..."
9,valence,valence_3NVLIWA4.csv,1000,6,6,"[Frame#, Time]","[Frame#, Time, Valence/class, Valence/probabil..."


In [94]:
for _, row in sample_files_by_type.iterrows():
    file_type = row["file_type"]
    file_path = Path(row["file_path"])
    
    df_sample = read_csv_auto(file_path, nrows=5)
    
    print("\n" + "=" * 80)
    print("FILE TYPE:", file_type)
    print("FILE NAME:", row["file_name"])
    print("SHAPE:", df_sample.shape)
    print("COLUMNS:")
    print(df_sample.columns.tolist())
    
    display(df_sample.head())


FILE TYPE: arousal
FILE NAME: arousal_3NVLIWA4.csv
SHAPE: (5, 6)
COLUMNS:
['Frame#', 'Time', 'Arousal/class', 'Arousal/probability', 'Imu/MotionIntensity', 'Ppg/QualityIndex']


,Frame#,Time,Arousal/class,Arousal/probability,Imu/MotionIntensity,Ppg/QualityIndex
0,1,1.703164e+09,0.0,0.59,0.066667,1.0
1,2,1.703164e+09,0.0,0.59,0.066667,1.0
2,3,1.703164e+09,0.0,0.59,0.066667,1.0
3,4,1.703164e+09,0.0,0.59,0.066667,1.0
4,5,1.703164e+09,0.0,0.59,0.066667,1.0



FILE TYPE: breathingRate
FILE NAME: breathingRate_3NVLIWA4.csv
SHAPE: (5, 5)
COLUMNS:
['Frame#', 'Time', 'BreathingRate', 'Ppg/QualityIndex', 'Imu/MotionIntensity']


,Frame#,Time,BreathingRate,Ppg/QualityIndex,Imu/MotionIntensity
0,1,1.703164e+09,13.102513,1.0,0.076923
1,2,1.703164e+09,13.102513,1.0,0.076923
2,3,1.703164e+09,13.102513,1.0,0.076923
3,4,1.703164e+09,13.102513,1.0,0.076923
4,5,1.703164e+09,13.102513,1.0,0.076923



FILE TYPE: emgActivation
FILE NAME: emgActivation_3NVLIWA4.csv
SHAPE: (5, 6)
COLUMNS:
['Frame#', 'Time', 'Emg/Amplitude/zygo/weighted', 'Emg/Amplitude/orbi/weighted', 'Emg/Amplitude/front/weighted', 'Emg/Amplitude/corr/weighted']


...

,Frame#,Time,HRV/mean_hr,HRV/rr,HRV/sdnn,HRV/sdsd,HRV/rmssd,Imu/MotionIntensity,Ppg/QualityIndex
0,1,1.703164e+09,84.862385,707.027027,86.269999,117.184038,117.189306,0.066667,1.0
1,2,1.703164e+09,84.862385,707.027027,86.269999,117.184038,117.189306,0.066667,1.0
2,3,1.703164e+09,84.862385,707.027027,86.269999,117.184038,117.189306,0.066667,1.0
3,4,1.703164e+09,84.862385,707.027027,86.269999,117.184038,117.189306,0.066667,1.0
4,5,1.703164e+09,84.862385,707.027027,86.269999,117.184038,117.189306,0.066667,1.0



FILE TYPE: raw_sensor_data
FILE NAME: 3NVLIWA4.csv
SHAPE: (5, 59)
COLUMNS:
['Frame#', 'Time', 'Faceplate/FaceState', 'Faceplate/FitState', 'Emg/ContactStates[RightOrbicularis]', 'Emg/Contact[RightOrbicularis]', 'Emg/Raw[RightOrbicularis]', 'Emg/RawLift[RightOrbicularis]', 'Emg/Filtered[RightOrbicularis]', 'Emg/Amplitude[RightOrbicularis]', 'Emg/ContactStates[RightZygomaticus]', 'Emg/Contact[RightZygomaticus]', 'Emg/Raw[RightZygomaticus]', 'Emg/RawLift[RightZygomaticus]', 'Emg/Filtered[RightZygomaticus]', 'Emg/Amplitude[RightZygomaticus]', 'Emg/ContactStates[RightFrontalis]', 'Emg/Contact[RightFrontalis]', 'Emg/Raw[RightFrontalis]', 'Emg/RawLift[RightFrontalis]', 'Emg/Filtered[RightFrontalis]', 'Emg/Amplitude[RightFrontalis]', 'Emg/ContactStates[CenterCorrugator]', 'Emg/Contact[CenterCorrugator]', 'Emg/Raw[CenterCorrugator]', 'Emg/RawLift[CenterCorrugator]', 'Emg/Filtered[CenterCorrugator]', 'Emg/Amplitude[CenterCorrugator]', 'Emg/ContactStates[LeftFrontalis]', 'Emg/Contact[LeftFrontal

,Frame#,Time,Faceplate/FaceState,Faceplate/FitState,Emg/ContactStates[RightOrbicularis],Emg/Contact[RightOrbicularis],Emg/Raw[RightOrbicularis],Emg/RawLift[RightOrbicularis],Emg/Filtered[RightOrbicularis],Emg/Amplitude[RightOrbicularis],...,Accelerometer/Raw.x,Accelerometer/Raw.y,Accelerometer/Raw.z,Magnetometer/Raw.x,Magnetometer/Raw.y,Magnetometer/Raw.z,Gyroscope/Raw.x,Gyroscope/Raw.y,Gyroscope/Raw.z,Pressure/Raw
0,1,1.703164e+09,1,9,255,0,342226,0,1205,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
1,2,1.703164e+09,1,9,255,0,328841,0,-639,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
2,3,1.703164e+09,1,9,255,0,342238,0,106,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
3,4,1.703164e+09,1,9,255,0,328938,0,-760,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
4,5,1.703164e+09,1,9,255,0,342534,0,289,1178,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416



FILE TYPE: valence
FILE NAME: valence_3NVLIWA4.csv
SHAPE: (5, 6)
COLUMNS:
['Frame#', 'Time', 'Valence/class', 'Valence/probability', 'Imu/MotionIntensity', 'Ppg/QualityIndex']


,Frame#,Time,Valence/class,Valence/probability,Imu/MotionIntensity,Ppg/QualityIndex
0,1,1.703164e+09,0.0,0.415,0.066667,1.0
1,2,1.703164e+09,0.0,0.415,0.066667,1.0
2,3,1.703164e+09,0.0,0.415,0.066667,1.0
3,4,1.703164e+09,0.0,0.415,0.066667,1.0
4,5,1.703164e+09,0.0,0.415,0.066667,1.0


In [95]:
raw_sample_path = Path(
    files_df[files_df["file_type"] == "raw_sensor_data"]
    .iloc[0]["file_path"]
)

raw_sample = read_csv_auto(raw_sample_path, nrows=5)

print("Shape:", raw_sample.shape)
print("Columns:")
print(raw_sample.columns.tolist()[:20])

raw_sample.head()

Shape: (5, 59)
Columns:
['Frame#', 'Time', 'Faceplate/FaceState', 'Faceplate/FitState', 'Emg/ContactStates[RightOrbicularis]', 'Emg/Contact[RightOrbicularis]', 'Emg/Raw[RightOrbicularis]', 'Emg/RawLift[RightOrbicularis]', 'Emg/Filtered[RightOrbicularis]', 'Emg/Amplitude[RightOrbicularis]', 'Emg/ContactStates[RightZygomaticus]', 'Emg/Contact[RightZygomaticus]', 'Emg/Raw[RightZygomaticus]', 'Emg/RawLift[RightZygomaticus]', 'Emg/Filtered[RightZygomaticus]', 'Emg/Amplitude[RightZygomaticus]', 'Emg/ContactStates[RightFrontalis]', 'Emg/Contact[RightFrontalis]', 'Emg/Raw[RightFrontalis]', 'Emg/RawLift[RightFrontalis]']


,Frame#,Time,Faceplate/FaceState,Faceplate/FitState,Emg/ContactStates[RightOrbicularis],Emg/Contact[RightOrbicularis],Emg/Raw[RightOrbicularis],Emg/RawLift[RightOrbicularis],Emg/Filtered[RightOrbicularis],Emg/Amplitude[RightOrbicularis],...,Accelerometer/Raw.x,Accelerometer/Raw.y,Accelerometer/Raw.z,Magnetometer/Raw.x,Magnetometer/Raw.y,Magnetometer/Raw.z,Gyroscope/Raw.x,Gyroscope/Raw.y,Gyroscope/Raw.z,Pressure/Raw
0,1,1.703164e+09,1,9,255,0,342226,0,1205,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
1,2,1.703164e+09,1,9,255,0,328841,0,-639,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
2,3,1.703164e+09,1,9,255,0,342238,0,106,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
3,4,1.703164e+09,1,9,255,0,328938,0,-760,0,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416
4,5,1.703164e+09,1,9,255,0,342534,0,289,1178,...,-1617,-1366,7810,0,0,0,-186,-295,-31,9719416


The inspection shows that the dataset contains multiple time-series CSV file types for each participant. Most files include `Frame#` and `Time` columns and contain numeric measurements suitable for min/max feature extraction. The raw sensor data files are the richest source of physiological features, while the derived files provide more compact affective and physiological signals.

The raw sensor files require special handling because they contain metadata rows at the beginning. After skipping these metadata rows, the files are read correctly with all sensor columns. The `headMotion` files contain very limited data, so they should be handled carefully or treated as optional during feature extraction.

The arousal and valence CSV files represent derived signal outputs and should not be confused with the questionnaire-based valence and arousal labels that will be merged later.

Based on the dataset description, the per-participant CSV files appear to represent two levels of data. The raw sensor data files contain recorded physiological and sensor variables such as EMG, PPG, heart rate, IMU, and pressure measurements.

The other CSV files appear to contain derived affective and physiological insights generated after further processing by the Emteq Emotion AI Engine. These include HRV, breathing rate, facial expression, arousal, valence, facial activation, facial valence, head motion, and EMG activation measures.

Therefore, the `arousal` and `valence` CSV files are treated as engine-derived signal outputs, not as the questionnaire-based target labels. The target labels will be taken from the questionnaire data.

## 5. Feature Column Selection

In this segment, the numeric columns that can be used for feature extraction are selected from the available per-participant CSV files.

The goal is to prepare a clean list of feature columns for each file type before calculating video-level statistics such as minimum and maximum values.

<b>Feature extraction will be performed mainly from the per-participant sensor CSV files. </b> These include both the main `raw_sensor_data` file and the derived signal files such as HRV, breathing rate, facial expression, facial activation, EMG activation, arousal, valence, and facial valence.

The structured questionnaire data will not be used as input sensor features. Instead, it will be used later for target labels such as questionnaire-based valence, arousal, discomfort, and empathy scores.

Columns such as `Frame#`, `Time`, class labels, status indicators, and quality-related columns will be handled carefully or excluded when they do not represent useful continuous sensor measurements.

This step helps prevent mixing input features with target labels and prepares the dataset for whole-video min/max feature extraction.

In [ ]:
'''
1. od sensor CSV fajlovi vadime min/max features 
2. Od details zemame koj participant, koj video/window
3. Od questionnaries gi zemame target labels
4. Na kraj merge vo edna video-level tabela

* arousal_.csv i valence_.csv se od sensor CSV fajlovi i ne se labeli, takvite csv od dekoj participant ke gi koristime kako input features
prvin ke napravime nekoj baseline model bez niv, a poto ai so niv
'''

In [ ]:
#bez ovaa funkcija ne mi dava tocni rezultati za cistenje
def clean_column_name(col):
    return (
        str(col)
        .replace("\ufeff", "")   # brise BOM ako postoi
        .replace("\u200b", "")   # brise zero-width space ako postoi
        .replace("\xa0", " ")    # brise non-breaking space
        .strip()
    )


# koi oloni ne mozat da se tretiraat kako features, tuku se metadata ili labeli
def should_exclude_column(col):
    col_clean = clean_column_name(col)
    col_lower = col_clean.lower()

    # Frame# i Time ne se features, tuku indeks/vreme
    if col_lower in ["frame#", "frame", "time"]:
        return True

    # Ovie koloni ne gi tretirame kako kontinuirani sensor features
    excluded_keywords = [
        "class",
        "state",
        "status",
        "quality",
        "contact",
        "fit"
    ]

    # Ako imeto na kolonata sodrzi nekoj od ovie zborovi, ja isklucuvame
    if any(keyword in col_lower for keyword in excluded_keywords):
        return True

    return False

feature_column_records = []

# izbor na koloni vrz sample fajlovi,
for _, row in sample_files_by_type.iterrows():

    file_type = row["file_type"]
    file_path = Path(row["file_path"])

    df_sample = read_csv_auto(file_path, nrows=1000)

    df_sample.columns = [clean_column_name(col) for col in df_sample.columns]

    numeric_columns = df_sample.select_dtypes(include=np.number).columns.tolist()

    selected_columns = []
    excluded_columns = []

    # Za sekoja numeric kolona proveruvame dali e feature ili ne
    for col in numeric_columns:
        if should_exclude_column(col):
            excluded_columns.append(col)
        else:
            selected_columns.append(col)

    feature_column_records.append({
        "file_type": file_type,
        "number_of_numeric_columns": len(numeric_columns),
        "number_of_selected_feature_columns": len(selected_columns),
        "selected_feature_columns": selected_columns,
        "excluded_numeric_columns": excluded_columns
    })

feature_columns_df = pd.DataFrame(feature_column_records)

feature_columns_df

,file_type,number_of_numeric_columns,number_of_selected_feature_columns,selected_feature_columns,excluded_numeric_columns
0,arousal,6,2,"[Arousal/probability, Imu/MotionIntensity]","[Frame#, Time, Arousal/class, Ppg/QualityIndex]"
1,breathingRate,5,2,"[BreathingRate, Imu/MotionIntensity]","[Frame#, Time, Ppg/QualityIndex]"
2,emgActivation,6,4,"[Emg/Amplitude/zygo/weighted, Emg/Amplitude/or...","[Frame#, Time]"
3,expression,3,1,[Expression/Intensity],"[Frame#, Time]"
4,facialActivation,3,1,[FacialActivation],"[Frame#, Time]"
5,facialValence,3,1,[FacialValence],"[Frame#, Time]"
6,headMotion,1,1,[headMotion],[]
7,hrv,9,6,"[HRV/mean_hr, HRV/rr, HRV/sdnn, HRV/sdsd, HRV/...","[Frame#, Time, Ppg/QualityIndex]"
8,raw_sensor_data,59,41,"[Emg/Raw[RightOrbicularis], Emg/RawLift[RightO...","[Frame#, Time, Faceplate/FaceState, Faceplate/..."
9,valence,6,2,"[Valence/probability, Imu/MotionIntensity]","[Frame#, Time, Valence/class, Ppg/QualityIndex]"


In [99]:
#test
for _, row in feature_columns_df.iterrows():
    print("\nFILE TYPE:", row["file_type"])
    print("SELECTED:")
    print(row["selected_feature_columns"])
    print("EXCLUDED:")
    print(row["excluded_numeric_columns"])



FILE TYPE: arousal
SELECTED:
['Arousal/probability', 'Imu/MotionIntensity']
EXCLUDED:
['Frame#', 'Time', 'Arousal/class', 'Ppg/QualityIndex']

FILE TYPE: breathingRate
SELECTED:
['BreathingRate', 'Imu/MotionIntensity']
EXCLUDED:
['Frame#', 'Time', 'Ppg/QualityIndex']

FILE TYPE: emgActivation
SELECTED:
['Emg/Amplitude/zygo/weighted', 'Emg/Amplitude/orbi/weighted', 'Emg/Amplitude/front/weighted', 'Emg/Amplitude/corr/weighted']
EXCLUDED:
['Frame#', 'Time']

FILE TYPE: expression
SELECTED:
['Expression/Intensity']
EXCLUDED:
['Frame#', 'Time']

FILE TYPE: facialActivation
SELECTED:
['FacialActivation']
EXCLUDED:
['Frame#', 'Time']

FILE TYPE: facialValence
SELECTED:
['FacialValence']
EXCLUDED:
['Frame#', 'Time']

FILE TYPE: headMotion
SELECTED:
['headMotion']
EXCLUDED:
[]

FILE TYPE: hrv
SELECTED:
['HRV/mean_hr', 'HRV/rr', 'HRV/sdnn', 'HRV/sdsd', 'HRV/rmssd', 'Imu/MotionIntensity']
EXCLUDED:
['Frame#', 'Time', 'Ppg/QualityIndex']

FILE TYPE: raw_sensor_data
SELECTED:
['Emg/Raw[RightOrbicu

In [100]:
# Od feature_columns_df pravime dictionary:
# key = file_type
# value = lista na koloni sto ke gi koristime kako features
#
# Primer:
# feature_columns_by_type["arousal"]
# ke vrati ['Arousal/probability', 'Imu/MotionIntensity']

feature_columns_by_type = {}

for _, row in feature_columns_df.iterrows():
    file_type = row["file_type"]
    selected_columns = row["selected_feature_columns"]
    
    feature_columns_by_type[file_type] = selected_columns

feature_columns_by_type

{'arousal': ['Arousal/probability', 'Imu/MotionIntensity'],
 'breathingRate': ['BreathingRate', 'Imu/MotionIntensity'],
 'emgActivation': ['Emg/Amplitude/zygo/weighted',
  'Emg/Amplitude/orbi/weighted',
  'Emg/Amplitude/front/weighted',
  'Emg/Amplitude/corr/weighted'],
 'expression': ['Expression/Intensity'],
 'facialActivation': ['FacialActivation'],
 'facialValence': ['FacialValence'],
 'headMotion': ['headMotion'],
 'hrv': ['HRV/mean_hr',
  'HRV/rr',
  'HRV/sdnn',
  'HRV/sdsd',
  'HRV/rmssd',
  'Imu/MotionIntensity'],
 'raw_sensor_data': ['Emg/Raw[RightOrbicularis]',
  'Emg/RawLift[RightOrbicularis]',
  'Emg/Filtered[RightOrbicularis]',
  'Emg/Amplitude[RightOrbicularis]',
  'Emg/Raw[RightZygomaticus]',
  'Emg/RawLift[RightZygomaticus]',
  'Emg/Filtered[RightZygomaticus]',
  'Emg/Amplitude[RightZygomaticus]',
  'Emg/Raw[RightFrontalis]',
  'Emg/RawLift[RightFrontalis]',
  'Emg/Filtered[RightFrontalis]',
  'Emg/Amplitude[RightFrontalis]',
  'Emg/Raw[CenterCorrugator]',
  'Emg/RawLif

In [101]:
# Proveruvame dali izbranite feature koloni postojat kaj site fajlovi
# Ova e vazno bidejki sample fajlot mozebi ima koloni koi ne postojat kaj nekoj drug participant

feature_availability_records = []

for _, row in files_df.iterrows():
    file_type = row["file_type"]
    file_path = Path(row["file_path"])
    
    if file_type not in feature_columns_by_type:
        continue
    
    selected_columns = feature_columns_by_type[file_type]
    
    try:
        df_sample = read_csv_auto(file_path, nrows=5)
        
        df_sample.columns = [clean_column_name(col) for col in df_sample.columns]
        
        available_columns = df_sample.columns.tolist()
        
        missing_columns = [
            col for col in selected_columns
            if col not in available_columns
        ]
        
        feature_availability_records.append({
            "participant_id": row["participant_id"],
            "file_type": file_type,
            "file_name": row["file_name"],
            "number_of_expected_features": len(selected_columns),
            "number_of_missing_features": len(missing_columns),
            "missing_features": missing_columns
        })
        
    except Exception as e:
        feature_availability_records.append({
            "participant_id": row["participant_id"],
            "file_type": file_type,
            "file_name": row["file_name"],
            "number_of_expected_features": len(selected_columns),
            "number_of_missing_features": None,
            "missing_features": str(e)
        })

feature_availability_df = pd.DataFrame(feature_availability_records)

feature_availability_df.head()

,participant_id,file_type,file_name,number_of_expected_features,number_of_missing_features,missing_features
0,1,raw_sensor_data,3NVLIWA4.csv,41,0,[]
1,1,arousal,arousal_3NVLIWA4.csv,2,0,[]
2,1,breathingRate,breathingRate_3NVLIWA4.csv,2,0,[]
3,1,emgActivation,emgActivation_3NVLIWA4.csv,4,0,[]
4,1,expression,expression_3NVLIWA4.csv,1,0,[]


In [103]:
# Summary za da vidime dali nekoj file_type ima problem so missing koloni

feature_availability_summary = (
    feature_availability_df
    .groupby("file_type")["number_of_missing_features"]
    .max()
    .reset_index()
    .sort_values("number_of_missing_features", ascending=False)
)

feature_availability_summary

,file_type,number_of_missing_features
0,arousal,0
1,breathingRate,0
2,emgActivation,0
3,expression,0
4,facialActivation,0
5,facialValence,0
6,headMotion,0
7,hrv,0
8,raw_sensor_data,0
9,valence,0


The feature availability check shows that all expected feature columns are present in the available CSV files. No missing feature columns were detected for any file type.

This means that the selected feature columns can be safely used for min/max extraction. However, this check only confirms column availability inside existing files; it does not mean that every participant has every CSV file type.

## 6. Video Window Information

In this section, the video start and end times are loaded from `final_data_details.csv`.

These time values will be used to filter each participant's sensor data so that min/max features are calculated only during the video-watching period.

In [104]:
details_path = data_dir / "final_data_details.csv"

details_df = read_csv_auto(details_path)

print("Details shape:", details_df.shape)
details_df.head()

Details shape: (105, 8)


,ID,start_cal_time_h,start_cal_time_m,start_vid_time_h,start_vid_time_m,end_vid_time_h,end_vid_time_m,additional info
0,1,14,1,14,5,14.0,36.0,work
1,2,13,6,13,10,13.0,38.0,work
2,3,14,11,14,15,14.0,35.0,work - without RC vid
3,4,13,9,13,13,13.0,34.0,work
4,5,14,18,14,22,14.0,47.0,work


In [105]:
for i, col in enumerate(details_df.columns):
    print(i, col)

0 ID
1 start_cal_time_h
2 start_cal_time_m
3 start_vid_time_h
4 start_vid_time_m
5 end_vid_time_h
6 end_vid_time_m
7 additional info


In [ ]:
# only the columns we need

video_windows_df = details_df[[
    "ID",
    "start_vid_time_h",
    "start_vid_time_m",
    "end_vid_time_h",
    "end_vid_time_m"
]].copy()

video_windows_df = video_windows_df.rename(columns={
    "ID": "participant_id"
})

video_windows_df.head()

,participant_id,start_vid_time_h,start_vid_time_m,end_vid_time_h,end_vid_time_m
0,1,14,5,14.0,36.0
1,2,13,10,13.0,38.0
2,3,14,15,14.0,35.0
3,4,13,13,13.0,34.0
4,5,14,22,14.0,47.0


In [ ]:
#calculating the time in minutes for start and end of video windows ?????????? ova ne e ok izlaga eka 20-30min gleda video, za videoto trae 2min?????????
#mozda se misli na celo snimanje, neka stoi ovoj cell pa ke vidime sto e

video_windows_df["video_start_minutes"] = (
    video_windows_df["start_vid_time_h"] * 60 +
    video_windows_df["start_vid_time_m"]
)

video_windows_df["video_end_minutes"] = (
    video_windows_df["end_vid_time_h"] * 60 +
    video_windows_df["end_vid_time_m"]
)

video_windows_df["video_duration_minutes"] = (
    video_windows_df["video_end_minutes"] -
    video_windows_df["video_start_minutes"]
)

video_windows_df.head()

,participant_id,start_vid_time_h,start_vid_time_m,end_vid_time_h,end_vid_time_m,video_start_minutes,video_end_minutes,video_duration_minutes
0,1,14,5,14.0,36.0,845,876.0,31.0
1,2,13,10,13.0,38.0,790,818.0,28.0
2,3,14,15,14.0,35.0,855,875.0,20.0
3,4,13,13,13.0,34.0,793,814.0,21.0
4,5,14,22,14.0,47.0,862,887.0,25.0


The questionnaires table contains separate self-reported labels for four videos per participant. The columns `valence1`–`valence4` and `arousal1`–`arousal4` correspond to the four video-level label values.

The start and end times from `final_data_details.csv` appear to describe the full video-watching session rather than a single video, because the calculated durations are much longer than two minutes. Therefore, additional information is needed to split the sensor data into four individual video windows before creating a true video-level dataset.

In [110]:
questionnaires_path = data_dir / "final_data_questionnaires.csv"
questionnaires_df = read_csv_auto(questionnaires_path)

label_rows = []

for _, row in questionnaires_df.iterrows():
    participant_id = row["ID"]

    for video_number in [1, 2, 3, 4]:
        label_rows.append({
            "participant_id": participant_id,
            "video_number": video_number,
            "valence": row[f"valence{video_number}"],
            "arousal": row[f"arousal{video_number}"]
        })

video_labels_df = pd.DataFrame(label_rows)

print("Shape:", video_labels_df.shape)
video_labels_df.head(12)

Shape: (420, 4)


,participant_id,video_number,valence,arousal
0,1,1,4,3
1,1,2,3,2
2,1,3,0,2
3,1,4,4,3
4,2,1,3,3
5,2,2,2,2
6,2,3,1,1
7,2,4,3,1
8,3,1,1,3
9,3,2,3,3


In [111]:
video_labels_df[["valence", "arousal"]].describe()

,valence,arousal
count,420.000000,420.000000
mean,2.361905,1.997619
std,1.197641,1.025914
min,0.000000,0.000000
25%,2.000000,1.000000
50%,3.000000,2.000000
75%,3.000000,3.000000
max,9.000000,4.000000


In [112]:
video_labels_df[["valence", "arousal"]].isna().sum()

valence    0
arousal    0
dtype: int64

In [113]:
video_labels_df["video_number"].value_counts().sort_index()

video_number
1    105
2    105
3    105
4    105
Name: count, dtype: int64

The video-level label table contains 420 rows, corresponding to 105 participants and 4 videos per participant. No missing values were found in the selected valence and arousal labels, and each video has exactly 105 label entries.

This confirms that the questionnaire-based labels are complete and ready to be merged later with the extracted video-level sensor features.

## 7. Video Window Approximation

The questionnaire data provides four separate valence and arousal labels for each participant, which means that the sensor data needs to be aligned with four video-level windows.

The start and end times from `final_data_details.csv` appear to describe the full video-watching session rather than individual video timestamps. Since exact start and end times for each separate video are not available in this notebook, the next step is to approximate four video windows by splitting each participant's full session window into four equal parts.

These approximated windows will then be used to calculate min/max sensor features for each participant and each video.

In [118]:
'''ova e so aproximate delenje

#create 4 approximate video windoes

video_window_rows = []

for _, row in video_windows_df.iterrows():
    participant_id = row["participant_id"]
    
    session_start = row["video_start_minutes"]
    session_end = row["video_end_minutes"]
    session_duration = session_end - session_start
    
    window_duration = session_duration / 4
    
    for video_number in [1, 2, 3, 4]:
        window_start = session_start + (video_number - 1) * window_duration
        window_end = session_start + video_number * window_duration
        
        video_window_rows.append({
            "participant_id": participant_id,
            "video_number": video_number,
            "window_start_minutes": window_start,
            "window_end_minutes": window_end,
            "window_duration_minutes": window_duration
        })

video_windows_long_df = pd.DataFrame(video_window_rows)

print("Shape:", video_windows_long_df.shape)
video_windows_long_df.head(12)
'''

video_window_rows = []

video_duration_minutes = 2

for _, row in video_windows_df.iterrows():
    participant_id = row["participant_id"]
    
    session_start = row["video_start_minutes"]
    session_end = row["video_end_minutes"]
    session_duration = session_end - session_start
    
    segment_duration = session_duration / 4
    
    for video_number in [1, 2, 3, 4]:
        segment_start = session_start + (video_number - 1) * segment_duration
        segment_end = session_start + video_number * segment_duration
        
        window_start = segment_start
        window_end = min(segment_start + video_duration_minutes, segment_end)
        
        video_window_rows.append({
            "participant_id": participant_id,
            "video_number": video_number,
            "segment_start_minutes": segment_start,
            "segment_end_minutes": segment_end,
            "window_start_minutes": window_start,
            "window_end_minutes": window_end,
            "window_duration_minutes": window_end - window_start
        })

video_windows_long_df = pd.DataFrame(video_window_rows)

print("Shape:", video_windows_long_df.shape)
video_windows_long_df.head(12)

Shape: (420, 7)


,participant_id,video_number,segment_start_minutes,segment_end_minutes,window_start_minutes,window_end_minutes,window_duration_minutes
0,1.0,1,845.00,852.75,845.00,847.00,2.0
1,1.0,2,852.75,860.50,852.75,854.75,2.0
2,1.0,3,860.50,868.25,860.50,862.50,2.0
3,1.0,4,868.25,876.00,868.25,870.25,2.0
4,2.0,1,790.00,797.00,790.00,792.00,2.0
5,2.0,2,797.00,804.00,797.00,799.00,2.0
6,2.0,3,804.00,811.00,804.00,806.00,2.0
7,2.0,4,811.00,818.00,811.00,813.00,2.0
8,3.0,1,855.00,860.00,855.00,857.00,2.0
9,3.0,2,860.00,865.00,860.00,862.00,2.0


In [119]:
#check window distribution
video_windows_long_df["video_number"].value_counts().sort_index()

video_number
1    105
2    105
3    105
4    105
Name: count, dtype: int64

In [120]:
video_windows_long_df[["window_start_minutes", "window_end_minutes", "window_duration_minutes"]].describe()

,window_start_minutes,window_end_minutes,window_duration_minutes
count,392.000000,392.000000,392.0
mean,876.653061,878.653061,2.0
std,87.761339,87.761339,0.0
min,692.000000,694.000000,2.0
25%,798.875000,800.875000,2.0
50%,870.625000,872.625000,2.0
75%,936.937500,938.937500,2.0
max,1107.750000,1109.750000,2.0


In [121]:
video_windows_long_df.isna().sum()

participant_id              0
video_number                0
segment_start_minutes      28
segment_end_minutes        28
window_start_minutes       28
window_end_minutes         28
window_duration_minutes    28
dtype: int64

In [122]:
video_windows_long_df[video_windows_long_df["window_start_minutes"].isna()]

,participant_id,video_number,segment_start_minutes,segment_end_minutes,window_start_minutes,window_end_minutes,window_duration_minutes
20,6.0,1,NaN,NaN,NaN,NaN,NaN
21,6.0,2,NaN,NaN,NaN,NaN,NaN
22,6.0,3,NaN,NaN,NaN,NaN,NaN
23,6.0,4,NaN,NaN,NaN,NaN,NaN
24,7.0,1,NaN,NaN,NaN,NaN,NaN
25,7.0,2,NaN,NaN,NaN,NaN,NaN
26,7.0,3,NaN,NaN,NaN,NaN,NaN
27,7.0,4,NaN,NaN,NaN,NaN,NaN
28,8.0,1,NaN,NaN,NaN,NaN,NaN
29,8.0,2,NaN,NaN,NaN,NaN,NaN


In [124]:
missing_window_participants = (
    video_windows_long_df[video_windows_long_df["window_start_minutes"].isna()]
    ["participant_id"]
    .drop_duplicates()
    .tolist()
)

missing_window_participants

[6.0, 7.0, 8.0, 9.0, 10.0, 11.0, 12.0]

Seven participants do not have valid session start and end times in `final_data_details.csv`. Since each participant has four video labels, this results in 28 participant-video rows without computed video windows.

These participants cannot be used for time-window-based feature extraction unless their session times are recovered from another source. For the current notebook, they will be excluded from the window-based feature extraction step.

In [125]:
video_windows_long_clean_df = (
    video_windows_long_df
    .dropna(subset=["window_start_minutes", "window_end_minutes"])
    .copy()
)

print("Original windows:", video_windows_long_df.shape)
print("Clean windows:", video_windows_long_clean_df.shape)

video_windows_long_clean_df.head()

Original windows: (420, 7)
Clean windows: (392, 7)


,participant_id,video_number,segment_start_minutes,segment_end_minutes,window_start_minutes,window_end_minutes,window_duration_minutes
0,1.0,1,845.00,852.75,845.00,847.00,2.0
1,1.0,2,852.75,860.50,852.75,854.75,2.0
2,1.0,3,860.50,868.25,860.50,862.50,2.0
3,1.0,4,868.25,876.00,868.25,870.25,2.0
4,2.0,1,790.00,797.00,790.00,792.00,2.0


### Approximation of Individual Video Windows

The dataset does not provide exact start and end timestamps for each individual video. Instead, `final_data_details.csv` contains the start and end time of the full video-watching session.

Two possible approaches were considered:

1. Splitting the full session into four equal parts, one for each video.  
   However, this produced windows of approximately 5 to 7 minutes per video, which is longer than the expected video duration. This suggests that the full session also includes pauses, transitions, or questionnaire time between videos.

2. Using fixed two-minute windows starting at the beginning of each approximated segment.  
   Since each video is expected to last approximately two minutes, this approach is more consistent with the actual video duration and avoids including longer non-video periods.

The second approach was selected. However, these windows are still approximate, because the exact timestamp of each video onset is not available. If a video started slightly after the beginning of the approximated segment, some noise may still be included. This limitation should be considered when interpreting the extracted features.

## 8. Video-Level Min/Max Feature Extraction

In this section, min and max values are extracted from the selected sensor feature columns for each participant and each approximated two-minute video window.

The extraction is performed only for participant-video rows that have both questionnaire labels and valid video window information.

In [126]:
#merge labels with clean windows
video_labels_with_windows_df = video_labels_df.merge(
    video_windows_long_clean_df,
    on=["participant_id", "video_number"],
    how="inner"
)

print("Labels shape:", video_labels_df.shape)
print("Labels with valid windows shape:", video_labels_with_windows_df.shape)

video_labels_with_windows_df.head(20)

Labels shape: (420, 4)
Labels with valid windows shape: (392, 9)


,participant_id,video_number,valence,arousal,segment_start_minutes,segment_end_minutes,window_start_minutes,window_end_minutes,window_duration_minutes
0,1,1,4,3,845.00,852.75,845.00,847.00,2.0
1,1,2,3,2,852.75,860.50,852.75,854.75,2.0
2,1,3,0,2,860.50,868.25,860.50,862.50,2.0
3,1,4,4,3,868.25,876.00,868.25,870.25,2.0
4,2,1,3,3,790.00,797.00,790.00,792.00,2.0
5,2,2,2,2,797.00,804.00,797.00,799.00,2.0
6,2,3,1,1,804.00,811.00,804.00,806.00,2.0
7,2,4,3,1,811.00,818.00,811.00,813.00,2.0
8,3,1,1,3,855.00,860.00,855.00,857.00,2.0
9,3,2,3,3,860.00,865.00,860.00,862.00,2.0


In [134]:
#convert sensor time into minutes
def add_time_minutes_column(df):
    if "Time" not in df.columns:
        return df
    
    df = df.copy()
    
    time_datetime = pd.to_datetime(
        df["Time"],
        unit="s",
        errors="coerce",
        utc=True
    ).dt.tz_convert("Europe/Skopje")
    
    df["time_minutes"] = (
        time_datetime.dt.hour * 60 +
        time_datetime.dt.minute +
        time_datetime.dt.second / 60
    )
    
    return df

In [132]:
#helper fo feature names

def make_feature_name(file_type, column_name, statistic):
    clean_name = (
        str(column_name)
        .replace("/", "_")
        .replace("[", "_")
        .replace("]", "")
        .replace(" ", "_")
        .replace("#", "number")
    )
    
    return f"{file_type}_{clean_name}_{statistic}"

In [135]:
raw_sample = read_csv_auto(raw_sample_path, nrows=1000)
raw_sample.columns = [clean_column_name(col) for col in raw_sample.columns]
raw_sample = add_time_minutes_column(raw_sample)

print("Sensor time range:")
print(raw_sample["time_minutes"].min())
print(raw_sample["time_minutes"].max())

print("Test windows:")
display(test_windows[["video_number", "window_start_minutes", "window_end_minutes"]])

Sensor time range:
841.8166666666667
841.8333333333334
Test windows:


,video_number,window_start_minutes,window_end_minutes
0,1,845.00,847.00
1,2,852.75,854.75
2,3,860.50,862.50
3,4,868.25,870.25


In [137]:
raw_sample_larger = read_csv_auto(raw_sample_path, nrows=300000)

raw_sample_larger.columns = [
    clean_column_name(col) for col in raw_sample_larger.columns
]

raw_sample_larger = add_time_minutes_column(raw_sample_larger)

print("Larger sensor time range:")
print(raw_sample_larger["time_minutes"].min())
print(raw_sample_larger["time_minutes"].max())

Larger sensor time range:
841.8166666666667
846.8166666666667


In [140]:
for _, window_row in test_windows.iterrows():
    video_number = window_row["video_number"]
    window_start = window_row["window_start_minutes"]
    window_end = window_row["window_end_minutes"]

    count_rows = raw_sample_larger[
        (raw_sample_larger["time_minutes"] >= window_start) &
        (raw_sample_larger["time_minutes"] < window_end)
    ].shape[0]

    print(video_number, window_start, window_end, "rows:", count_rows)

1.0 845.0 847.0 rows: 109780
2.0 852.75 854.75 rows: 0
3.0 860.5 862.5 rows: 0
4.0 868.25 870.25 rows: 0


In [139]:
window_1_df = raw_sample_larger[
    (raw_sample_larger["time_minutes"] >= 845.0) &
    (raw_sample_larger["time_minutes"] < 847.0)
]

print("Window 1 shape:", window_1_df.shape)

test_col = "HeartRate/Average"

print(window_1_df[test_col].describe())


Window 1 shape: (109780, 60)
count    109780.000000
mean         94.948149
std           3.417898
min          89.700000
25%          91.140000
50%          96.730000
75%          97.890000
max          99.510000
Name: HeartRate/Average, dtype: float64


In [141]:
#test extraction on one participant first

test_participant_id = video_labels_with_windows_df["participant_id"].iloc[0]

test_windows = video_labels_with_windows_df[
    video_labels_with_windows_df["participant_id"] == test_participant_id
]

test_files = files_df[
    files_df["participant_id"] == test_participant_id
]

feature_rows = []

for _, window_row in test_windows.iterrows():
    participant_id = window_row["participant_id"]
    video_number = window_row["video_number"]
    window_start = window_row["window_start_minutes"]
    window_end = window_row["window_end_minutes"]
    
    feature_row = {
        "participant_id": participant_id,
        "video_number": video_number
    }
    
    for _, file_row in test_files.iterrows():
        file_type = file_row["file_type"]
        
        if file_type not in feature_columns_by_type:
            continue
        
        selected_columns = feature_columns_by_type[file_type]
        file_path = Path(file_row["file_path"])
        
        df = read_csv_auto(file_path)
        df.columns = [clean_column_name(col) for col in df.columns]
        df = add_time_minutes_column(df)
        
        if "time_minutes" not in df.columns:
            continue
        
        window_df = df[
            (df["time_minutes"] >= window_start) &
            (df["time_minutes"] < window_end)
        ]
        
        for col in selected_columns:
            if col not in window_df.columns:
                continue
            
            feature_row[make_feature_name(file_type, col, "min")] = window_df[col].min()
            feature_row[make_feature_name(file_type, col, "max")] = window_df[col].max()
    
    feature_rows.append(feature_row)

test_features_df = pd.DataFrame(feature_rows)

print("Test feature shape:", test_features_df.shape)
test_features_df.head()

Test feature shape: (4, 122)


,participant_id,video_number,raw_sensor_data_Emg_Raw_RightOrbicularis_min,raw_sensor_data_Emg_Raw_RightOrbicularis_max,raw_sensor_data_Emg_RawLift_RightOrbicularis_min,raw_sensor_data_Emg_RawLift_RightOrbicularis_max,raw_sensor_data_Emg_Filtered_RightOrbicularis_min,raw_sensor_data_Emg_Filtered_RightOrbicularis_max,raw_sensor_data_Emg_Amplitude_RightOrbicularis_min,raw_sensor_data_Emg_Amplitude_RightOrbicularis_max,...,hrv_HRV_sdsd_min,hrv_HRV_sdsd_max,hrv_HRV_rmssd_min,hrv_HRV_rmssd_max,hrv_Imu_MotionIntensity_min,hrv_Imu_MotionIntensity_max,valence_Valence_probability_min,valence_Valence_probability_max,valence_Imu_MotionIntensity_min,valence_Imu_MotionIntensity_max
0,1.0,1.0,362290.0,463254.0,0.0,0.0,-3280.0,2843.0,22.0,1116.0,...,79.276776,244.528701,79.282497,244.528701,0.0,0.400000,0.370,0.605,0.0,0.400000
1,1.0,2.0,545946.0,560887.0,0.0,0.0,-3175.0,2593.0,22.0,1080.0,...,68.912592,140.947884,69.012975,140.960163,0.0,0.000000,0.390,0.690,0.0,0.000000
2,1.0,3.0,577894.0,613510.0,0.0,0.0,-798.0,1103.0,21.0,326.0,...,70.992957,164.996350,70.992957,165.042493,0.0,0.066667,0.345,0.715,0.0,0.066667
3,1.0,4.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


<b> !!!!!!!!!! Ova mora da se smeni, zs so window 2 min poslednito segment ne go fakja i e NaN !!!!!!!!!!!</b>

## 9. Final Video-Level Min/Max Feature Extraction

In this step, the selected sensor feature columns are extracted for every valid participant-video window.  
For each participant and each approximated 2-minute video window, minimum and maximum values are calculated from the available sensor CSV files.

The extraction is performed only for participant-video rows that have valid window information and valid questionnaire labels.

In [142]:
import time

def extract_video_level_minmax_features(
    labels_with_windows_df,
    files_df,
    feature_columns_by_type,
    checkpoint_every=10
):
    all_feature_rows = []
    
    participant_ids = sorted(
        labels_with_windows_df["participant_id"]
        .dropna()
        .astype(int)
        .unique()
    )
    
    start_time = time.time()
    
    for idx, participant_id in enumerate(participant_ids, start=1):
        participant_windows = labels_with_windows_df[
            labels_with_windows_df["participant_id"].astype(int) == participant_id
        ].copy()
        
        participant_files = files_df[
            files_df["participant_id"].astype(int) == participant_id
        ].copy()
        
        participant_feature_rows = {}
        
        for _, window_row in participant_windows.iterrows():
            video_number = int(window_row["video_number"])
            
            participant_feature_rows[video_number] = {
                "participant_id": participant_id,
                "video_number": video_number,
                "valence": window_row["valence"],
                "arousal": window_row["arousal"],
                "window_start_minutes": window_row["window_start_minutes"],
                "window_end_minutes": window_row["window_end_minutes"],
                "window_duration_minutes": window_row["window_duration_minutes"]
            }
        
        for _, file_row in participant_files.iterrows():
            file_type = file_row["file_type"]
            
            if file_type not in feature_columns_by_type:
                continue
            
            selected_columns = feature_columns_by_type[file_type]
            file_path = Path(file_row["file_path"])
            
            try:
                df = read_csv_auto(file_path)
            except Exception as e:
                print(f"Could not read file: {file_path.name} | Participant: {participant_id} | Error: {e}")
                continue
            
            df.columns = [clean_column_name(col) for col in df.columns]
            df = add_time_minutes_column(df)
            
            if "time_minutes" not in df.columns:
                continue
            
            available_columns = [
                col for col in selected_columns
                if col in df.columns
            ]
            
            if len(available_columns) == 0:
                continue
            
            for _, window_row in participant_windows.iterrows():
                video_number = int(window_row["video_number"])
                window_start = window_row["window_start_minutes"]
                window_end = window_row["window_end_minutes"]
                
                window_df = df[
                    (df["time_minutes"] >= window_start) &
                    (df["time_minutes"] < window_end)
                ]
                
                for col in available_columns:
                    values = pd.to_numeric(window_df[col], errors="coerce")
                    
                    participant_feature_rows[video_number][
                        make_feature_name(file_type, col, "min")
                    ] = values.min()
                    
                    participant_feature_rows[video_number][
                        make_feature_name(file_type, col, "max")
                    ] = values.max()
        
        all_feature_rows.extend(participant_feature_rows.values())
        
        if idx % checkpoint_every == 0 or idx == len(participant_ids):
            elapsed = round((time.time() - start_time) / 60, 2)
            print(
                f"Processed {idx}/{len(participant_ids)} participants "
                f"| rows so far: {len(all_feature_rows)} "
                f"| elapsed: {elapsed} min"
            )
    
    return pd.DataFrame(all_feature_rows)

In [143]:
video_level_minmax_df = extract_video_level_minmax_features(
    labels_with_windows_df=video_labels_with_windows_df,
    files_df=files_df,
    feature_columns_by_type=feature_columns_by_type,
    checkpoint_every=10
)

print("Final feature dataset shape:", video_level_minmax_df.shape)
video_level_minmax_df.head()

KeyboardInterrupt: 

In [ ]:
output_path = data_dir / "video_level_valence_arousal_minmax.csv"

video_level_minmax_df.to_csv(output_path, index=False)

print("Saved final dataset to:")
print(output_path)